In [33]:
!nvidia-smi


Wed Mar  4 23:02:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.97                 Driver Version: 580.97         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060      WDDM  |   00000000:2B:00.0  On |                  N/A |
|  0%   41C    P1             22W /  145W |    2124MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [221]:
import yaml
import os
from ultralytics import YOLO

# ── МОДЕЛЬ ───────────────────────────────────────────────────────
# На 130 фото 'n' (nano) не выучит детали, а 'x' не влезет. 
# 'm' (medium) — идеальный баланс точности для 8ГБ VRAM.
model = YOLO('yolo11m-seg.pt') 

base_path   = r'C:\Users\outlo\Desktop\all_data'
WEIGHTS_DIR = os.path.join(base_path, 'yolo_weights')

# ── YAML ────────────────────────────────────────────────────────
data_config = {
    'path':  base_path,
    'train': 'train/images',
    'val':   'train/images',
    'nc':    3,
    'names': {0: 'root', 1: 'stem', 2: 'leaf'}
}
yaml_path = os.path.join(base_path, 'custom_data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
    
results = model.train(
    data=yaml_path,
    epochs=200,         # ↑ увеличил, так как 100 для сегментации на малом датасете мало
    patience=50,
    imgsz=960,          # ↑ 960 — компромисс между 640 и 1280. 5060 потянет батч 4.
    batch=4,            
    device=0,
    name='plants_optimized_seg',
    project=WEIGHTS_DIR,

    # ── Оптимизатор ─────────────────────────────────────────────
    optimizer='AdamW',
    lr0=0.001,          # Стандартный lr для AdamW лучше сходится
    cos_lr=True,        # Включил косинусное затухание для плавного финиша
    close_mosaic=10,    # Отключает мозаику (если включена) в конце обучения
    weight_decay=0.0005, # Регуляризация, чтобы модель не зазубривала фото

    # ── Аугментации (Тонкая настройка для корней) ────────────────
    mosaic=0.5,         # ↓ Снизил до 0.5. Совсем выключать нельзя — упадет обобщение.
    copy_paste=0.6,     # ↑ Поднял. Для сегментации корней это лучшая аугментация.
    mixup=0.1,          # Чуть-чуть добавим для устойчивости к шуму
    shear=5.0,
    
    flipud=0.5,          # Корни могут расти и вверх, и вниз (отражение по вертикали)
    hsv_h=0.015,

    # Геометрия: корни могут быть под любым углом
    degrees=15.0,       # ↑ Увеличил угол, корни в земле не всегда вертикальны
    scale=0.5,          # Разные масштабы помогут видеть и тонкие, и толстые корни

    # ── Качество масок ───────────────────────────────────────────
    # ВНИМАНИЕ: mask_ratio=1 потребляет МНОГО памяти. 
    # Если упадет с OOM — верните на default (4) или поставьте 2.
    mask_ratio=1,       
    retina_masks=True,  
    overlap_mask=True,

    # ── Технические параметры ────────────────────────────────────
    workers=4,          # Оптимально для 5060, чтобы не забивать шину данных
    exist_ok=True,
    deterministic=False # Выключил для небольшого ускорения
)


New https://pypi.org/project/ultralytics/8.4.20 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.6, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\outlo\Desktop\all_data\custom_data.yaml, degrees=15.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.1, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=0.5, mul

C:\Users\outlo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 2490.41092.6 MB/s, size: 595.9 KB)
val: Scanning C:\Users\outlo\Desktop\all_data\train\labels.cache... 147 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 147/147  0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 115 weight(decay=0.0), 126 weight(decay=0.0005), 125 bias(decay=0.0)
Plotting labels to C:\Users\outlo\Desktop\all_data\yolo_weights\plants_optimized_seg\labels.jpg... 
Image sizes 960 train, 960 val
Using 4 dataloader workers
Logging results to C:\Users\outlo\Desktop\all_data\yolo_weights\plants_optimized_seg
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/200      10.1G      1.926      3.293      4

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.08 GiB. GPU 0 has a total capacity of 7.96 GiB of which 0 bytes is free. Of the allocated memory 8.17 GiB is allocated by PyTorch, and 1.59 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import cv2
import numpy as np
import networkx as nx
import os
import glob
from ultralytics import YOLO
from skimage.morphology import skeletonize, remove_small_objects
from skimage.measure import label, regionprops
from scipy.spatial import cKDTree

# ─── SETTINGS ────────────────────────────────────────────────────────────────
ROOT_DIR = r'C:\Users\outlo\Desktop\all_data'
SEG_WEIGHTS = os.path.join(ROOT_DIR, 'yolo_weights', 'plants_optimized_seg', 'weights', 'best.pt')
SAVE_DIR = os.path.join(ROOT_DIR, 'inference_results', 'graphs_v11_en_clean')
DEBUG_DIR = os.path.join(SAVE_DIR, 'debug')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

# Classes
ROOT_CLASSES = {'root', 'корень'}
STEM_CLASSES = {'stem', 'стебель'}
LEAF_CLASSES  = {'leaf', 'лист'}

MASK_OVERLAP_THRESH  = 0.60
COMBINED_MIN_SIZE    = 60
SKELETON_MIN_LENGTH  = 30

GRAPH_CONNECT_RADIUS = 1.7
GRAPH_MAX_NODES      = 38000

MAX_SHORT_BRANCH_LEN_ROOT = 1.42

# Colors
COLORS = {
    'root':  {'edge': (255, 0, 180), 'node_end': (0, 0, 255),   'node_mid': (180, 0, 255)},
    'stem':  {'edge': (0, 220, 0),   'node_end': (0, 100, 0),   'node_mid': (100, 255, 100)},
    'leaf':  {'edge': (0, 180, 255), 'node_end': (0, 80, 180),  'node_mid': (100, 220, 255)},
}

model = YOLO(SEG_WEIGHTS)

img_path = max(glob.glob(os.path.join(ROOT_DIR, 'seek', '*.[jJ][pP][gG]')), key=os.path.getmtime)
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Cannot load image: {img_path}")

# ─── после загрузки изображения ────────────────────────────────
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Cannot load image: {img_path}")

gray_global = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)          # ← добавить эту строку


def keep_best_one_per_class(results):
    """
    Оставляет по одному объекту на каждый класс: самый уверенный + большой.
    Предполагаем, что классы: root, stem, leaf (можно поменять имена/индексы)
    """
    if len(results.boxes) == 0:
        return results
    
    # Достаём данные
    boxes = results.boxes
    confs = boxes.conf.cpu().numpy()                  # уверенности
    classes = boxes.cls.cpu().numpy().astype(int)     # индексы классов
    # Площадь по bounding box (xyxy -> width*height)
    areas = ((boxes.xyxy[:, 2] - boxes.xyxy[:, 0]) * 
             (boxes.xyxy[:, 3] - boxes.xyxy[:, 1])).cpu().numpy()
    
    # Если есть маски — можно использовать их площадь вместо боксов (часто точнее)
    if results.masks is not None:
        mask_areas = np.array([m.sum().item() for m in results.masks.data])
        # Можно заменить areas на mask_areas, если хочешь считать по реальной сегментации
        # areas = mask_areas
    
    # Нормализуем площади (0..1)
    if areas.max() > 0:
        norm_areas = areas / areas.max()
    else:
        norm_areas = np.zeros_like(areas)
    
    # Комбинированный скор: conf чуть важнее, но размер тоже учитываем
    combined_scores = confs ** 1.2 * (norm_areas ** 0.8 + 1e-6)  # +eps чтобы не было нуля
    
    # Группируем по классам и выбираем лучший в каждой группе
    keep_indices = []
    unique_classes = np.unique(classes)
    
    for cls in unique_classes:
        mask = (classes == cls)
        if mask.sum() == 0:
            continue
        cls_scores = combined_scores[mask]
        best_local_idx = cls_scores.argmax()
        # Глобальный индекс лучшего в этом классе
        global_idx = np.where(mask)[0][best_local_idx]
        keep_indices.append(global_idx)
    
    if not keep_indices:
        # ничего не нашли — очищаем
        results.boxes = results.boxes[:0]
        if results.masks is not None:
            results.masks = results.masks[:0]
        return results
    
    # Сортируем индексы (опционально, чтобы порядок был как в оригинале)
    keep_indices = sorted(keep_indices)
    
    # Оставляем только выбранные
    results.boxes = results.boxes[keep_indices]
    if results.masks is not None:
        results.masks = results.masks[keep_indices]
    # Если есть другие атрибуты (probs, obb, keypoints и т.д.) — тоже обрезай аналогично
    
    return results


# Пример использования:
results = model.predict(source=img, conf=0.10, imgsz=960, retina_masks=True)[0]
results = keep_best_one_per_class(results)

# Теперь в results будет максимум по 1 объекту на класс (root, stem, leaf)
print(f"Осталось объектов: {len(results.boxes)}")

# ─── HELPER FUNCTIONS ────────────────────────────────────────────────────────

def get_adaptive_params(gray, mask):
    if not np.any(mask):
        return 35, 9
    masked_gray = gray[mask]
    if len(masked_gray) < 100:
        return 25, 7
    contrast = np.std(masked_gray)
    block_size = int(25 + 0.05 * np.sqrt(mask.sum()))
    block_size = block_size if block_size % 2 == 1 else block_size + 1
    block_size = max(15, min(95, block_size))
    C = max(1, min(12, 9 - contrast / 15))
    return block_size, C

def adaptive_binarize(gray, mask_bool):
    if not np.any(mask_bool):
        return np.zeros_like(gray, dtype=bool)
    block, C = get_adaptive_params(gray, mask_bool)
    masked = gray.copy()
    masked[~mask_bool] = 255
    binary = cv2.adaptiveThreshold(masked, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, blockSize=block, C=C)
    binary = (binary == 0) & mask_bool
    binary = remove_small_objects(binary, min_size=COMBINED_MIN_SIZE)
    return binary

def filter_skeleton_by_mask(skeleton, seg_mask, thresh=0.5):
    if thresh <= 0 or not skeleton.any():
        return skeleton
    labeled = label(skeleton)
    out = np.zeros_like(skeleton, dtype=bool)
    for lbl in range(1, labeled.max() + 1):
        comp = (labeled == lbl)
        total = comp.sum()
        if total < SKELETON_MIN_LENGTH: continue
        inside = (comp & seg_mask).sum()
        if inside / total >= thresh: out |= comp
    return out

def keep_longest_component(skeleton):
    if not skeleton.any(): return skeleton
    labeled = label(skeleton)
    props = regionprops(labeled)
    if not props: return np.zeros_like(skeleton, dtype=bool)
    longest = max(props, key=lambda x: x.area)
    return labeled == longest.label

def build_graph(skeleton, keep_only_largest=False):
    yx = np.column_stack(np.where(skeleton))
    if len(yx) == 0: return nx.Graph(), skeleton
    step = max(1, len(yx) // GRAPH_MAX_NODES)
    yx = yx[::step]
    if len(yx) == 0: return nx.Graph(), skeleton
    tree = cKDTree(yx)
    pairs = tree.query_pairs(r=GRAPH_CONNECT_RADIUS)
    G = nx.Graph()
    node_map = {i: (int(yx[i][1]), int(yx[i][0])) for i in range(len(yx))}
    for xy in node_map.values(): G.add_node(xy)
    for i, j in pairs:
        n1, n2 = node_map[i], node_map[j]
        dist = float(np.linalg.norm(yx[i] - yx[j]))
        G.add_edge(n1, n2, weight=dist)
    if len(G) and keep_only_largest:
        largest = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest).copy()
    return G, skeleton

def remove_short_terminal_branches(G, max_len):
    if max_len <= 0:
        return
    changed = True
    while changed:
        changed = False
        to_remove = []
        for node in list(G.nodes()):
            if G.degree(node) == 1:
                nbrs = list(G.neighbors(node))
                if len(nbrs) == 1:
                    nbr = nbrs[0]
                    if G[node][nbr]['weight'] <= max_len:
                        to_remove.append((node, nbr))
        if to_remove:
            G.remove_edges_from(to_remove)
            G.remove_nodes_from(list(nx.isolates(G)))
            changed = True

def get_color(cls_group):
    return COLORS.get(cls_group, COLORS['root'])

def make_bbox_mask(shape, x1, y1, x2, y2):
    mask = np.zeros(shape[:2], dtype=bool)
    mask[y1:y2, x1:x2] = True
    return mask

def save_debug_stages(name_prefix, base_crop, stages):
    if not stages: return
    h, w = base_crop.shape[:2]
    total_height = h * len(stages) + 40 * (len(stages) + 1)
    debug_canvas = np.ones((total_height, w, 3), dtype=np.uint8) * 30
    y_offset = 30
    for title, stage_img in stages:
        if stage_img is None or stage_img.size == 0: continue
        if stage_img.dtype == bool: stage_img = stage_img.astype(np.uint8) * 255
        if len(stage_img.shape) == 2: stage_img = cv2.cvtColor(stage_img, cv2.COLOR_GRAY2BGR)
        if stage_img.shape[:2] != (h, w): stage_img = cv2.resize(stage_img, (w, h), interpolation=cv2.INTER_AREA)
        debug_canvas[y_offset:y_offset + h, 0:w] = stage_img
        cv2.putText(debug_canvas, title, (15, y_offset + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        y_offset += h + 40
    path = os.path.join(DEBUG_DIR, f"DEBUG_{name_prefix}_stages.jpg")
    cv2.imwrite(path, debug_canvas)

# ─── MASK COLLECTION + AREA CALCULATION ──────────────────────────────────────
vis_img = img.copy()
masks_by_group = {
    'leaf': np.zeros(img.shape[:2], dtype=bool),
    'stem': np.zeros(img.shape[:2], dtype=bool),
    'root': []
}

bbox_by_group = {'leaf': None, 'stem': None}

# Сбор масок и сразу считаем площадь
areas = {'root': 0, 'stem': 0, 'leaf': 0}

for idx, (mask_tensor, box) in enumerate(zip(results.masks.data, results.boxes.xyxy)):
    cls_name = model.names[int(results.boxes.cls[idx])].lower()
    seg_mask = mask_tensor.cpu().numpy() > 0.5
    x1, y1, x2, y2 = map(int, box.cpu().numpy())

    if cls_name in ROOT_CLASSES:
        bbox_mask = make_bbox_mask(img.shape, x1, y1, x2, y2)
        masks_by_group['root'].append((bbox_mask, (x1, y1, x2, y2), seg_mask))
        areas['root'] += np.sum(seg_mask)
    elif cls_name in STEM_CLASSES:
        masks_by_group['stem'] |= seg_mask
        areas['stem'] += np.sum(seg_mask)
        if bbox_by_group['stem'] is None:
            bbox_by_group['stem'] = (x1, y1, x2, y2)
        else:
            ex1, ey1, ex2, ey2 = bbox_by_group['stem']
            bbox_by_group['stem'] = (min(ex1, x1), min(ey1, y1), max(ex2, x2), max(ey2, y2))
    elif cls_name in LEAF_CLASSES:
        masks_by_group['leaf'] |= seg_mask
        areas['leaf'] += np.sum(seg_mask)
        if bbox_by_group['leaf'] is None:
            bbox_by_group['leaf'] = (x1, y1, x2, y2)
        else:
            ex1, ey1, ex2, ey2 = bbox_by_group['leaf']
            bbox_by_group['leaf'] = (min(ex1, x1), min(ey1, y1), max(ex2, x2), max(ey2, y2))

def keep_longest_component(skeleton):
    if not skeleton.any():
        return skeleton
    labeled = label(skeleton)
    props = regionprops(labeled)
    if not props:
        return np.zeros_like(skeleton, dtype=bool)
    longest = max(props, key=lambda x: x.area)
    return labeled == longest.label

def connect_nearby_components(G, max_dist=25, min_dist_to_connect=3.0):
    """
    Соединяет ближайшие концевые точки разных компонент, если расстояние в разумных пределах
    """
    if nx.is_connected(G):
        return G

    components = list(nx.connected_components(G))
    if len(components) <= 1:
        return G

    # Собираем только концы (degree 1) каждой компоненты
    tips_per_comp = {}
    for i, comp in enumerate(components):
        subgraph = G.subgraph(comp)
        tips = [n for n in subgraph if subgraph.degree(n) == 1]
        if not tips:  # если компонента без концов (цикл) — берём любой узел
            tips = list(comp)[:1]
        tips_per_comp[i] = tips

    # Все концы + их координаты
    all_tips = []
    for i, tips in tips_per_comp.items():
        for tip in tips:
            all_tips.append((tip, i))  # (координата, номер компоненты)

    if len(all_tips) < 2:
        return G

    coords = np.array([pt for pt, _ in all_tips])
    tree = cKDTree(coords)

    # Ищем пары близких концов из РАЗНЫХ компонент
    to_add = []
    used = set()

    pairs = tree.query_pairs(r=max_dist)
    for idx1, idx2 in pairs:
        pt1, comp1 = all_tips[idx1]
        pt2, comp2 = all_tips[idx2]

        if comp1 == comp2:
            continue  # один компонент — не соединяем

        dist = np.linalg.norm(np.array(pt1) - np.array(pt2))
        if dist > max_dist or dist < min_dist_to_connect:
            continue

        # Простое правило: соединяем, если никто из них ещё не использован
        if idx1 not in used and idx2 not in used:
            to_add.append((pt1, pt2, dist))
            used.add(idx1)
            used.add(idx2)

    # Добавляем рёбра (можно сортировать по dist, если хочешь самые короткие первыми)
    for u, v, d in sorted(to_add, key=lambda x: x[2]):
        G.add_edge(u, v, weight=d)

    return G

# Вывод площадей в консоль
print("\n" + "="*60)
print("  AREAS (pixels)")
print("="*60)
print(f"  Roots total area     : {areas['root']:,} px²")
print(f"  Stem total area      : {areas['stem']:,} px²")
print(f"  Leaf total area      : {areas['leaf']:,} px²")
print(f"  Total plant area     : {areas['root'] + areas['stem'] + areas['leaf']:,} px²")
print("="*60 + "\n")


# ─── PRINT OTHER SETTINGS ────────────────────────────────────────────────────
print("=" * 70)
print("  GRAPH PARAMETERS")
print("=" * 70)
print(f"    MASK_OVERLAP_THRESH         = {MASK_OVERLAP_THRESH}")
print(f"    COMBINED_MIN_SIZE           = {COMBINED_MIN_SIZE} px")
print(f"    SKELETON_MIN_LENGTH         = {SKELETON_MIN_LENGTH} px")
print(f"    MAX_SHORT_BRANCH_LEN_ROOT   = {MAX_SHORT_BRANCH_LEN_ROOT} px (только root)")
print(f"    GRAPH_CONNECT_RADIUS        = {GRAPH_CONNECT_RADIUS}")
print("=" * 70)

# ─── PROCESSING ──────────────────────────────────────────────────────────────


# Stem
if masks_by_group['stem'].any():
    print("\n▶ [stem] STEM   longest component")

    crop_orig_stem = img.copy()

    stages_stem = []
    stages_stem.append(("Original", crop_orig_stem.copy()))
    stages_stem.append(("Binary mask", masks_by_group['stem']))

    skeleton = skeletonize(masks_by_group['stem'])
    stages_stem.append(("Full skeleton", skeleton))

    longest_skel = keep_longest_component(skeleton)
    stages_stem.append(("Longest component", longest_skel))

    G, _ = build_graph(longest_skel)

    G = connect_nearby_components(G, max_dist=100, min_dist_to_connect=4.0)
    print(f"   nodes: {G.number_of_nodes():4d}   edges: {G.number_of_edges():4d}")

    crop_graph = crop_orig_stem.copy()
    color = get_color('stem')
    total_len = sum(d['weight'] for _,_,d in G.edges(data=True))

    for u, v, d in G.edges(data=True):
        cv2.line(crop_graph, u, v, color['edge'], 2)

    for node in G.nodes():
        if G.degree(node) != 2:
            cv2.circle(crop_graph, node, 6, color['node_end'], -1)
        else:
            cv2.circle(crop_graph, node, 3, color['node_mid'], -1)

    stages_stem.append((f"Final graph  L:{int(total_len)} px", crop_graph))

    save_debug_stages("stem", crop_orig_stem, stages_stem)

    for u, v, d in G.edges(data=True):
        cv2.line(vis_img, u, v, color['edge'], 2)
    for node in G.nodes():
        if G.degree(node) != 2:
            cv2.circle(vis_img, node, 5, color['node_end'], -1)
        else:
            cv2.circle(vis_img, node, 2, color['node_mid'], -1)

    if bbox_by_group['stem']:
        x1, y1, x2, y2 = bbox_by_group['stem']
        info = f"stem | L:{int(total_len)} px | N:{G.number_of_nodes()}"
        cv2.putText(vis_img, info, (x1, max(y1 - 25, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)


# Leaf
if masks_by_group['leaf'].any():
    print("\n▶ [leaf] LEAF   longest component")

    crop_orig_leaf = img.copy()

    stages_leaf = []
    stages_leaf.append(("Original", crop_orig_leaf.copy()))
    stages_leaf.append(("Binary mask", masks_by_group['leaf']))

    skeleton = skeletonize(masks_by_group['leaf'])
    stages_leaf.append(("Full skeleton", skeleton))

    longest_skel = keep_longest_component(skeleton)
    stages_leaf.append(("Longest component", longest_skel))

    G, _ = build_graph(longest_skel)
    
    G = connect_nearby_components(G, max_dist=100, min_dist_to_connect=4.0)
    print(f"   nodes: {G.number_of_nodes():4d}   edges: {G.number_of_edges():4d}")

    crop_graph = crop_orig_leaf.copy()
    color = get_color('leaf')
    total_len = sum(d['weight'] for _,_,d in G.edges(data=True))

    for u, v, d in G.edges(data=True):
        cv2.line(crop_graph, u, v, color['edge'], 2)

    for node in G.nodes():
        if G.degree(node) != 2:
            cv2.circle(crop_graph, node, 6, color['node_end'], -1)
        else:
            cv2.circle(crop_graph, node, 3, color['node_mid'], -1)

    stages_leaf.append((f"Final graph  L:{int(total_len)} px", crop_graph))

    save_debug_stages("leaf", crop_orig_leaf, stages_leaf)

    for u, v, d in G.edges(data=True):
        cv2.line(vis_img, u, v, color['edge'], 2)
    for node in G.nodes():
        if G.degree(node) != 2:
            cv2.circle(vis_img, node, 5, color['node_end'], -1)
        else:
            cv2.circle(vis_img, node, 2, color['node_mid'], -1)

    if bbox_by_group['leaf']:
        x1, y1, x2, y2 = bbox_by_group['leaf']
        info = f"leaf | L:{int(total_len)} px | N:{G.number_of_nodes()}"
        cv2.putText(vis_img, info, (x1, max(y1 - 25, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)


# Roots
for root_idx, (bbox_mask, bbox, seg_mask) in enumerate(masks_by_group['root']):
    print(f"\n▶ [root-{root_idx}] ROOT   filter ≥{int(MASK_OVERLAP_THRESH*100)}%")

    x1, y1, x2, y2 = bbox
    pad = 100
    cy1 = max(0, y1 - pad)
    cy2 = min(img.shape[0], y2 + pad)
    cx1 = max(0, x1 - pad)
    cx2 = min(img.shape[1], x2 + pad)
    crop_orig = img[cy1:cy2, cx1:cx2].copy()

    stages = []
    stages.append(("Original (crop)", crop_orig.copy()))

    binary = adaptive_binarize(gray_global, bbox_mask)
    stages.append(("Adaptive threshold", binary))

    combined = binary & seg_mask
    stages.append(("After YOLO mask", combined))

    skeleton_raw = skeletonize(combined)
    stages.append(("Raw skeleton", skeleton_raw))

    skeleton_filt = filter_skeleton_by_mask(skeleton_raw, seg_mask, MASK_OVERLAP_THRESH)
    stages.append(("Filtered skeleton", skeleton_filt))

    G, _ = build_graph(skeleton_filt)

    to_remove_mask = [(u,v) for u,v in G.edges() if not (seg_mask[u[1],u[0]] and seg_mask[v[1],v[0]])]
    G.remove_edges_from(to_remove_mask)

    remove_short_terminal_branches(G, MAX_SHORT_BRANCH_LEN_ROOT)

    G.remove_nodes_from(list(nx.isolates(G)))
    
    G = connect_nearby_components(G, max_dist=100, min_dist_to_connect=4.0)
    print(f"   nodes: {G.number_of_nodes():4d}   edges: {G.number_of_edges():4d}")


    crop_graph = crop_orig.copy()
    color = get_color('root')
    total_len = sum(d['weight'] for _,_,d in G.edges(data=True))

    for u, v, d in G.edges(data=True):
        u_adj = (u[0] - cx1, u[1] - cy1)
        v_adj = (v[0] - cx1, v[1] - cy1)
        cv2.line(crop_graph, u_adj, v_adj, color['edge'], 2)

    for node in G.nodes():
        pt = (node[0] - cx1, node[1] - cy1)
        if G.degree(node) != 2:
            cv2.circle(crop_graph, pt, 4, color['node_end'], -1)
        else:
            cv2.circle(crop_graph, pt, 2, color['node_mid'], -1)

    stages.append((f"Final graph  L:{int(total_len)} px", crop_graph))

    save_debug_stages(f"root-{root_idx}", crop_orig, stages)

    # Draw on main image
    for u, v, d in G.edges(data=True):
        cv2.line(vis_img, u, v, color['edge'], 2)

    for node in G.nodes():
        if G.degree(node) != 2:
            cv2.circle(vis_img, node, 4, color['node_end'], -1)
        else:
            cv2.circle(vis_img, node, 2, color['node_mid'], -1)

    info = f"root-{root_idx} | L:{int(total_len)} px | N:{G.number_of_nodes()}"
    cv2.putText(vis_img, info, (x1, max(y1 - 25, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 140, 255), 2)

    
# ─── ОТРИСОВКА ПЛОЩАДЕЙ И МАСОК ──────────────────────────────────────────────

# Создаем копию для наложения масок (overlay)
overlay = vis_img.copy()

# Настройки для текста
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1.2
thickness = 3

# Проходим по группам для закраски
for group_name, mask_bool in [('leaf', masks_by_group['leaf']), 
                              ('stem', masks_by_group['stem'])]:
    if np.any(mask_bool):
        color = COLORS[group_name]['edge']
        # Закрашиваем область цветом (BGR)
        overlay[mask_bool] = color
        
        # Находим центр маски для вывода текста
        props = regionprops(label(mask_bool))
        if props:
            # Берем самую крупную область для метки
            main_prop = max(props, key=lambda x: x.area)
            cy, cx = main_prop.centroid
            txt = f"{group_name.upper()}: {areas[group_name]}px"

# Отдельная обработка для корней (так как они в списке)
root_full_mask = np.zeros(img.shape[:2], dtype=bool)
for bbox_m, coords, seg_m in masks_by_group['root']:
    root_full_mask |= seg_m

if np.any(root_full_mask):
    color = COLORS['root']['edge']
    overlay[root_full_mask] = color
    
    props = regionprops(label(root_full_mask))
    if props:
        main_prop = max(props, key=lambda x: x.area)
        cy, cx = main_prop.centroid
        txt = f"ROOTS: {areas['root']}px"

# Смешиваем оригинал и закраску (альфа 0.3 — прозрачность 30%)
alpha = 0.3
cv2.addWeighted(overlay, alpha, vis_img, 1 - alpha, 0, vis_img)


# ─── ОТРИСОВКА ПАНЕЛИ СТАТИСТИКИ В УГЛУ ─────────────────────────────────────

# Параметры панели
padding = 20
line_height = 40
panel_width = 450
panel_height = line_height * 4 + padding
margin = 30

# Создаем полупрозрачную подложку для читаемости
overlay = vis_img.copy()
cv2.rectangle(overlay, (margin, margin), (margin + panel_width, margin + panel_height), (0, 0, 0), -1)
cv2.addWeighted(overlay, 0.6, vis_img, 0.4, 0, vis_img)

# Список данных для вывода
stats = [
    (f"Roots Area: {areas['root']:,} px", COLORS['root']['edge']),
    (f"Stem Area:  {areas['stem']:,} px", COLORS['stem']['edge']),
    (f"Leaf Area:  {areas['leaf']:,} px", COLORS['leaf']['edge']),
    (f"Total Area: {sum(areas.values()):,} px", (255, 255, 255))
]

# Рисуем текст построчно
for i, (text, color) in enumerate(stats):
    y_pos = margin + padding + (i * line_height) + 10
    # Тонкая черная обводка для четкости
    cv2.putText(vis_img, text, (margin + 15, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 
                0.9, (0, 0, 0), 4, cv2.LINE_AA)
    # Основной текст (цвет соответствует категории)
    cv2.putText(vis_img, text, (margin + 15, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 
                0.9, color, 2, cv2.LINE_AA)

print(f"Статистика площадей отрисована в углу изображения.")


# ─── SAVE RESULT ─────────────────────────────────────────────────────────────
save_path = os.path.join(SAVE_DIR, f'graph_v11_en_clean_{os.path.basename(img_path)}')
cv2.imwrite(save_path, vis_img)
print(f"\nResult saved: {save_path}")
print(f"Debug images: {DEBUG_DIR}")


0: 736x960 1 root, 1 stem, 2 leafs, 237.7ms
Speed: 18.6ms preprocess, 237.7ms inference, 5.0ms postprocess per image at shape (1, 3, 736, 960)
Осталось объектов: 3

  AREAS (pixels)
  Roots total area     : 15,764 px²
  Stem total area      : 6,341 px²
  Leaf total area      : 12,547 px²
  Total plant area     : 34,652 px²

  GRAPH PARAMETERS
    MASK_OVERLAP_THRESH         = 0.6
    COMBINED_MIN_SIZE           = 60 px
    SKELETON_MIN_LENGTH         = 30 px
    MAX_SHORT_BRANCH_LEN_ROOT   = 1.5 px (только root)
    GRAPH_CONNECT_RADIUS        = 1.7

▶ [stem] STEM   longest component
   nodes:  339   edges:  338

▶ [leaf] LEAF   longest component
   nodes:  812   edges:  811

▶ [root-0] ROOT   filter ≥60%
   nodes:  762   edges:  800
Статистика площадей отрисована в углу изображения.

Result saved: C:\Users\outlo\Desktop\all_data\inference_results\graphs_v11_en_clean\graph_v11_en_clean_wheat_20260219151134536.jpg
Debug images: C:\Users\outlo\Desktop\all_data\inference_results\graphs_v